In [0]:
import time
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")
t0 = time.time()
plan = (
    orders
    .filter(F.col("o_orderdate") >= "1915-01-01")
    .withColumn("year", F.year("o_orderdate"))
    .groupBy("year")
    .agg(F.sum("o_totalprice").alias("revenue"))
)
print(f"Transformations took {time.time() - t0:.3f}s")

t0 = time.time()
display(plan)
print(f"Action took {time.time() - t0:.3f}s")
# `year` is not a column in `orders`, but is created with .withColumn before groupBy

In [0]:


def partition_sizes(df):
    """Rows per Partition. Serverless-safe: no RDD API."""
    return(
        df.groupBy(F.spark_partition_id().alias("partiotion_id"))
          .count()
          .orderBy("partiotion_id")
          )
    
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")

display(partition_sizes(orders))                                   # (i) as read
display(partition_sizes(orders.repartition(8)))                    # (ii) spread evenly
display(partition_sizes(orders.repartition(8, "o_orderstatus")))   # (iii) by key

In [0]:
customer = spark.table("samples.tpch.customer")
nation   = spark.table("samples.tpch.nation")
orders = spark.table("samples.tpch.orders")

narrow = orders.filter("o_orderstatus = 'P'").select("o_orderkey", "o_totalprice")
narrow.explain()

top_nations = (
    orders
    .join(customer, orders.o_custkey == customer.c_custkey)
    .join(nation, customer.c_nationkey == nation.n_nationkey)
    .groupBy("n_name")
    .agg(F.sum("o_totalprice").alias("total_revenue"))
    .orderBy(F.desc("total_revenue"))
)
top_nations.explain()

In [0]:
from pyspark.sql import functions as F

orders = spark.table("samples.tpch.orders")

bomb = orders.withColumn("boom", F.col("o_totalprice") / F.lit(0))   # divide by zero
print("Step 1: no error. Nothing has run.")

print(bomb.count())   # Step 2: does this fail?

bomb.select("boom").show(3)   # Step 3: does this fail?